# TRACER — Module 3: Dataset Management

Downloads real Flickr8k data, loads it through the `src/dataset` package built in Module 3,
splits it into train/val/test, and visualizes samples with their captions.

Self-contained — clones the repo automatically, works in a fresh Colab session with
Runtime → Run all.

In [ ]:
GITHUB_USERNAME = "IqraYasmin123"   # change if this isn't your username
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/tracers.git"

import os
if not os.path.exists("/content/tracers"):
    !git clone {REPO_URL} /content/tracers
else:
    %cd /content/tracers
    !git pull

%cd /content/tracers/ai-engine
import sys
sys.path.insert(0, ".")

from src.dataset import FlickrDataset, DatasetConfig, DatasetLoadError, show_samples
print("Imports OK.")


## Download Flickr8k

Uses a well-known GitHub-hosted mirror of Flickr8k (images + captions), so no Kaggle account
or API key is needed. Persisted to Google Drive so you don't have to re-download every
session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_DIR = "/content/drive/MyDrive/TRACER/datasets/flickr8k"
os.makedirs(DATASET_DIR, exist_ok=True)

images_marker = os.path.join(DATASET_DIR, "Images")
if not os.path.exists(images_marker):
    print("Downloading Flickr8k (images + text)... this takes a few minutes the first time.")
    !wget -q -O /content/Flickr8k_Dataset.zip https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
    !wget -q -O /content/Flickr8k_text.zip https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip
    !unzip -q -o /content/Flickr8k_Dataset.zip -d {DATASET_DIR}
    !unzip -q -o /content/Flickr8k_text.zip -d {DATASET_DIR}
    print("Download and extraction complete.")
else:
    print("Flickr8k already present in Drive — skipping download.")

!ls {DATASET_DIR}


## Load the dataset through FlickrDataset

If this raises `DatasetLoadError`, check the printed folder listing above against what
`FlickrDataset` expects (see `docs/module3_dataset.md`) — the error message tells you exactly
what it was looking for.

In [ ]:
config = DatasetConfig(
    root_dir=DATASET_DIR,
    image_size=(224, 224),
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42,
)

try:
    full_dataset = FlickrDataset(config)
    print(f"Loaded {len(full_dataset)} (image, caption) pairs.")
except DatasetLoadError as e:
    print(f"Dataset failed to load: {e}")


## Split into train / validation / test

In [ ]:
train_ds = full_dataset.get_split("train")
val_ds = full_dataset.get_split("val")
test_ds = full_dataset.get_split("test")

print(f"Train: {len(train_ds)}")
print(f"Val:   {len(val_ds)}")
print(f"Test:  {len(test_ds)}")
print(f"Total: {len(train_ds) + len(val_ds) + len(test_ds)} (should match {len(full_dataset)})")


## Visualize a sample grid with captions

In [ ]:
fig = show_samples(train_ds, n=6, cols=3)


## Inspect one sample directly

In [ ]:
sample = train_ds[0]
print(f"ID: {sample['id']}")
print(f"Caption: {sample['caption']}")
print(f"Image size: {sample['image'].size}")
sample['image']


## Module 3 completion check

Run the full test suite (no network needed — these test against fake data, not the real
download above):

```bash
cd ai-engine
pytest tests/test_dataset.py -v
```

All 14 tests should pass. Combined with the real Flickr8k load, split, and visualization
above working correctly, Module 3 is complete. Continue to
**Module 4 — Adversarial Attack Generation**.